## 自定义状态架构
#### 中间件可以使用自定义属性扩展代理的状态。定义自定义状态类型并将其设置为 ：state_schema

In [8]:
from langchain.agents.middleware import AgentState, AgentMiddleware
from typing_extensions import NotRequired
from langchain.messages import HumanMessage
from langchain.agents import create_agent
from typing import Any

class CustomState(AgentState):
    model_call_count: NotRequired[int]
    user_id: NotRequired[str]

class CallCounterMiddleware(AgentMiddleware[CustomState]):
    state_schema = CustomState

    def before_model(self, state: CustomState, runtime) -> dict[str, Any] | None:
        # Access custom state properties
        count = state.get("model_call_count", 0)

        if count > 10:
            return {"jump_to": "end"}

        return None

    def after_model(self, state: CustomState, runtime) -> dict[str, Any] | None:
        # Update custom state
        return {"model_call_count": state.get("model_call_count", 0) + 1}

agent = create_agent(
    model=model,
    middleware=[CallCounterMiddleware()],
    #tools=[...],
)

# Invoke with custom state
result = agent.invoke({
    "messages": [HumanMessage("Hello")],
    "model_call_count": 0,
    "user_id": "user-123",
})

print(result)

{'messages': [HumanMessage(content='Hello', additional_kwargs={}, response_metadata={}, id='73bafd87-5c50-42c9-9932-93ba684a44fd'), AIMessage(content='Hello! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 8, 'total_tokens': 18, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_efad92c60b', 'id': 'chatcmpl-CaGyGOzJ7Erc6heNuYC0d4p9TRzpx', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--b1dd79b3-51f8-4ec0-b3eb-d845ae3a9829-0', usage_metadata={'input_tokens': 8, 'output_tokens': 10, 'total_tokens': 18, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})], 'model_call_count': 1, 'user_i

## 执行顺序
#### 使用多个中间件时，了解执行顺序很重要：
```python
agent = create_agent(
    model="gpt-4o",
    middleware=[middleware1, middleware2, middleware3],
    tools=[...],
)
```
#### 它的正确执行过程为
##### 在钩子按顺序运行之前：
##### middleware1.before_agent()
##### middleware2.before_agent()
##### middleware3.before_agent()
##### 代理循环启动：
##### middleware1.before_model()
##### middleware2.before_model()
##### middleware3.before_model()
##### Wrap 钩子嵌套，类似于函数调用：
##### middleware1.wrap_model_call()→ → →模型
##### middleware2.wrap_model_call()
##### middleware3.wrap_model_call()
##### 钩子以相反的顺序运行后：
##### middleware3.after_model()
##### middleware2.after_model()
##### middleware1.after_model()
##### 代理循环结束：
##### middleware3.after_agent()
##### middleware2.after_agent()
##### middleware1.after_agent()

#### 官方文档还能进行直接跳转，
```python
class EarlyExitMiddleware(AgentMiddleware):
    def before_model(self, state: AgentState, runtime) -> dict[str, Any] | None:
        # Check some condition
        if should_exit(state):
            return {
                "messages": [AIMessage("Exiting early due to condition.")],
                "jump_to": "end"
            }
        return None
```
#### 可用的跳跃目标：
##### 'end'：跳转到代理执行的末尾
##### 'tools'：跳转到工具节点
##### 'model'：跳转到模型节点（或第一个钩子）before_model
#### 但我感觉除了end可用外，其余尽量不用，避免某一环节没跑通导致效果不好，且难以维护，具体的使用方法可以看官方文档的中间件最后部分